# 1. Imports

In [ ]:
!pip install accelerate diffusers controlnet_aux

In [ ]:
import os
from pathlib import Path
from PIL import Image
import torch
from controlnet_aux import CannyDetector
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
from diffusers.utils import load_image, make_image_grid

**ControlNet (Canny)** preserves the edges/structure of the content image. <br>
**IP-Adapter** takes care of the color palette, texture.




# 2. Configurations

In [ ]:
IMAGES_NUMBER = 3
INFERENCE_STEPS_NUMBER = 20
MAX_SIDE_IMG_SIZE = 768

# 3. Image Preprocessing

In [ ]:
def resize_img(img: Image.Image, max_side: int = 768, multiple_of: int = 8):
    w, h = img.size
    scale = min(1.0, max_side / max(w, h))
    new_w, new_h = round(w * scale / multiple_of) * multiple_of, round(h * scale / multiple_of) * multiple_of
    if (new_w, new_h) == (w, h):
        return img

    img = img.resize((new_w, new_h), Image.LANCZOS)
    return img

# 4. Edge Detection with CannyDetector

In [ ]:

canny_detector = CannyDetector()
canny_img = canny_detector(content_img, detect_resolution=min(content_img.size), image_resolution=min(content_img.size))

# 5. Models

In [ ]:
controlnet = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-canny",
                                             torch_dtype=torch.float16,
                                             use_safetensors=True)

pipeline = StableDiffusionControlNetPipeline.from_pretrained("Yntec/AbsoluteReality",
                                                         controlnet=controlnet,
                                                         torch_dtype=torch.float16)



In [ ]:
weight_name = "ip-adapter_sd15.bin"
weight_name=["ip-adapter_sd15.bin", "ip-adapter_sd15.bin"]  
pipeline.load_ip_adapter("h94/IP-Adapter", subfolder="models", weight_name=weight_name)


In [ ]:
ip_scale = IP_SCALE
ip_scale = [IP_SCALE, IP_SCALE]
pipeline.set_ip_adapter_scale(ip_scale)
pipeline.enable_model_cpu_offload()

# 6. Stylise

In [ ]:
negative_prompt = "low quality, blurry, disorted, deformed"
content_width, content_height = content_img.size

In [ ]:
ip_adapter_images = style_img
ip_adapter_images = [style_img, style_img2]

In [ ]:
stylised_images = pipeline(
    prompt=prompt,
    negative_prompt=negative_prompt,
    height=content_height,
    width=content_width,
    ip_adapter_image=ip_adapter_images,
    image=canny_img,
    guidance_scale=GUIDANCE_SCALE,
    controlnet_conditioning_scale=CONTROLNET_SCALE,
    num_inference_steps=INFERENCE_STEPS_NUMBER,
    num_images_per_prompt=IMAGES_NUMBER
).images

In [ ]:
make_image_grid(stylised_images, cols=3, rows=1)

In [ ]:
prompt = "fluffy dog, high quality, masterpiece"
content_img = load_image("../00_input_data/images/01_content/puppy.jpg")
content_img = resize_img(content_img)
style_img = load_image("../00_input_data/images/02_style/starry_night.jpg")
style_img2 = load_image("../00_input_data/images/02_style/circles_paint.jpg")

PARAMS:

**- guidance_scale:** [5 - 10] how strong to follow the text prompt;<br>
**- controlnet_conditioning_scale:** [0.5 - 1.0] lower scale if too much content leaking. If losing too much of the original structure - raise;<br>
**- ip_scale:** [0.4 - 0.8] set_ip_adapter_scale() raise if style not strong. If prompt ignored - lower;<br>

In [ ]:
IP_SCALE = 0.5 # lower value - prompt has more influence; higher value - style image has more influence
CONTROLNET_SCALE = 0.7
GUIDANCE_SCALE = 6 